# 4. Hypothesis Testing

We consider a search for a signal by **counting events**, where signal and background events are characterised by a variable $x$ ($0 \le x \le 1$) with PDFs:

$$f(x|s) = 3(1-x)^2 \qquad \text{(signal, peaked at low } x\text{)}$$
$$f(x|b) = 3x^2 \qquad \text{(background, peaked at high } x\text{)}$$

As a first step we **test the background hypothesis for each event**: reject the background hypothesis if $x < x_\text{cut}$.

The exercises below cover:
1. Critical region boundary $x_\text{cut}$ for a given test size $\alpha$
2. Power of the test with respect to the signal hypothesis
3. Expected numbers of selected signal and background events
4. Signal purity after the cut
5. $p$-value of the background-only hypothesis given $n$ observed events
6. Expected discovery significance $\text{median}[Z_b | s+b]$ and the $x_\text{cut}$ that maximises it
7. Monte Carlo investigation of the test of $s = 0$ using the $x$ values of each event (no cut)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import poisson, norm
from scipy.optimize import minimize_scalar
import pyhf

rng = np.random.default_rng(seed=123)

## Signal and background PDFs

We first visualise the two PDFs to build intuition about the discriminating power of $x$.

In [ ]:
x = np.linspace(0, 1, 500)

f_sig = 3 * (1 - x)**2   # f(x|s)
f_bkg = 3 * x**2          # f(x|b)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x, f_sig, label=r'$f(x|s) = 3(1-x)^2$', color='royalblue', lw=2)
ax.plot(x, f_bkg, label=r'$f(x|b) = 3x^2$',     color='tomato',    lw=2)
ax.set_xlabel('x')
ax.set_ylabel('PDF')
ax.set_title('Signal and background PDFs')
ax.legend()
plt.tight_layout()
plt.show()

## 1. Critical region boundary $x_\text{cut}$

We reject the background hypothesis for an event if $x < x_\text{cut}$.  
The **size** of the test (significance level $\alpha$) is the probability of rejecting the background hypothesis when the event truly is background:

$$\alpha = P(x < x_\text{cut} \mid b) = \int_0^{x_\text{cut}} 3x^2\, dx = x_\text{cut}^3$$

Inverting, the critical boundary corresponding to a chosen size $\alpha$ is:

$$x_\text{cut} = \alpha^{1/3}$$

In [ ]:
alpha = 0.05   # 5 % test size
x_cut = alpha ** (1.0 / 3.0)

print(f"Test size  alpha = {alpha}")
print(f"Critical boundary  x_cut = alpha^(1/3) = {x_cut:.4f}")

# Verify analytically
alpha_check = x_cut**3
print(f"Verification: x_cut^3 = {alpha_check:.4f}  (should equal alpha = {alpha})")

## 2. Power of the test

The **power** is the probability of correctly rejecting the background hypothesis when the event is signal:

$$1 - \beta = P(x < x_\text{cut} \mid s) = \int_0^{x_\text{cut}} 3(1-x)^2\, dx = 1 - (1 - x_\text{cut})^3$$

In [ ]:
power = 1.0 - (1.0 - x_cut)**3

print(f"x_cut = {x_cut:.4f}")
print(f"Power  1-beta = P(x < x_cut | s) = {power:.4f}")
print(f"Miss probability  beta = {1 - power:.4f}")

## 3. Expected numbers of selected events

In an experiment with $s_\text{tot} = 10$ signal events and $b_\text{tot} = 100$ background events, we apply a cut $x_\text{cut} = 0.1$.

The selection efficiencies are:
$$\varepsilon_s = P(x < x_\text{cut}\mid s) = 1 - (1 - x_\text{cut})^3$$
$$\varepsilon_b = P(x < x_\text{cut}\mid b) = x_\text{cut}^3$$

Expected selected counts:
$$n_s^{\text{exp}} = s_\text{tot}\cdot\varepsilon_s, \qquad n_b^{\text{exp}} = b_\text{tot}\cdot\varepsilon_b$$

In [ ]:
s_tot   = 10
b_tot   = 100
x_cut3  = 0.1    # fixed cut for parts 3-5

eps_s = 1.0 - (1.0 - x_cut3)**3
eps_b = x_cut3**3

n_s_exp = s_tot * eps_s
n_b_exp = b_tot * eps_b

print(f"x_cut = {x_cut3}")
print(f"Signal efficiency     eps_s = {eps_s:.4f}")
print(f"Background efficiency eps_b = {eps_b:.4f}")
print(f"Expected selected signal     n_s = s_tot * eps_s = {n_s_exp:.4f}")
print(f"Expected selected background n_b = b_tot * eps_b = {n_b_exp:.4f}")

## 4. Signal purity

The **signal purity** is the fraction of selected events that are true signal:

$$P = \frac{n_s^{\text{exp}}}{n_s^{\text{exp}} + n_b^{\text{exp}}}$$

In [ ]:
purity = n_s_exp / (n_s_exp + n_b_exp)

print(f"Signal purity  P = n_s / (n_s + n_b) = {purity:.4f}")

## 5. $p$-value of the background-only hypothesis

Suppose $n$ events are found with $x < x_\text{cut}$.  Under the **background-only** hypothesis ($s = 0$), the number of events in the signal region follows a Poisson distribution with mean $\mu_b = b_\text{tot} \cdot \varepsilon_b$:

$$p = P(N \ge n \mid \mu_b) = 1 - F_\text{Poisson}(n-1;\, \mu_b) = \sum_{k=n}^{\infty} \frac{e^{-\mu_b}\mu_b^k}{k!}$$

We compute this for a range of $n$ and highlight the discovery threshold $p < 2.87 \times 10^{-7}$ ($Z = 5\sigma$).

In [ ]:
mu_b = n_b_exp   # expected background in signal region

n_values = np.arange(0, 15)
p_values = poisson.sf(n_values - 1, mu_b)   # P(N >= n | mu_b)

print(f"Expected background in signal region: mu_b = {mu_b:.4f}")
print()
print(f"{'n':>4}  {'p-value':>14}  {'Z (sigma)':>10}")
print('-' * 35)
for n, p in zip(n_values, p_values):
    if p > 0:
        Z = norm.isf(p)
        print(f"{n:>4}  {p:>14.4e}  {Z:>10.3f}")
    else:
        print(f"{n:>4}  {'< 1e-300':>14}  {'> ~37':>10}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

mask = p_values > 0
Z_vals = np.where(mask, norm.isf(np.clip(p_values, 1e-300, 1)), np.nan)

axes[0].semilogy(n_values[mask], p_values[mask], 'o-', color='royalblue')
axes[0].axhline(2.87e-7, color='tomato', ls='--', label=r'$5\sigma$ threshold')
axes[0].set_xlabel('n observed events')
axes[0].set_ylabel('p-value')
axes[0].set_title(r'$p$-value vs. observed $n$')
axes[0].legend()

axes[1].plot(n_values[mask], Z_vals[mask], 'o-', color='royalblue')
axes[1].axhline(5, color='tomato', ls='--', label=r'$5\sigma$')
axes[1].set_xlabel('n observed events')
axes[1].set_ylabel('Significance Z')
axes[1].set_title('Significance vs. observed n')
axes[1].legend()

plt.tight_layout()
plt.show()

## 6. Expected discovery significance $\text{median}[Z_b | s+b]$

Under the signal+background hypothesis, the **median** observed count equals its expectation $n_s + n_b$.  The expected discovery significance is therefore:

$$Z_\text{med} = \Phi^{-1}\!\left(1 - P(N \ge n_s + n_b \mid n_b)\right)$$

Equivalently, using the asymptotic approximation (Cowan *et al.* 2011):

$$Z_\text{med} \approx \sqrt{2\left[(n_s+n_b)\ln\!\left(\frac{n_s+n_b}{n_b}\right) - n_s\right]}$$

We scan over $x_\text{cut}$ to find the value that maximises $Z_\text{med}$.

In [ ]:
def expected_significance(x_cut_val, s_total=10, b_total=100):
    """Asymptotic median discovery significance for a given x_cut."""
    n_s = s_total * (1.0 - (1.0 - x_cut_val)**3)
    n_b = b_total * x_cut_val**3
    if n_b <= 0 or n_s <= 0:
        return 0.0
    # Asymptotic formula (Cowan et al. 2011)
    Z = np.sqrt(2.0 * ((n_s + n_b) * np.log((n_s + n_b) / n_b) - n_s))
    return Z


x_scan = np.linspace(0.01, 0.99, 500)
Z_scan = np.array([expected_significance(xc) for xc in x_scan])

# Find optimal x_cut
result = minimize_scalar(
    lambda xc: -expected_significance(xc),
    bounds=(0.01, 0.99),
    method='bounded'
)
x_opt = result.x
Z_opt = expected_significance(x_opt)

print(f"Optimal x_cut = {x_opt:.4f}")
print(f"Maximum Z_med = {Z_opt:.4f}")
print()
print(f"At x_cut = 0.10 (from parts 3-5):")
print(f"  Z_med = {expected_significance(0.10):.4f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(x_scan, Z_scan, color='royalblue', lw=2, label=r'$Z_\mathrm{med}(x_\mathrm{cut})$')
ax.axvline(x_opt, color='tomato', ls='--', label=f'Optimal $x_{{\\rm cut}} = {x_opt:.3f}$')
ax.set_xlabel(r'$x_{\rm cut}$')
ax.set_ylabel(r'$Z_{\rm med}$')
ax.set_title(r'Expected discovery significance vs. $x_{\rm cut}$')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Monte Carlo investigation: testing $s = 0$ using individual $x$ values (no cut)

Instead of applying a cut, we use the **full information** in each event's $x$ value.  For each event, the likelihood ratio between the signal+background and background-only hypotheses is:

$$\lambda_i = \frac{f_\text{mix}(x_i)}{f(x_i|b)}, \qquad f_\text{mix}(x) = p_s\, f(x|s) + p_b\, f(x|b)$$

where $p_s = s_\text{tot}/(s_\text{tot}+b_\text{tot})$ and $p_b = b_\text{tot}/(s_\text{tot}+b_\text{tot})$ are the signal and background fractions.

The event-level **log-likelihood ratio** test statistic for $N$ observed events is:

$$t = -2\sum_{i=1}^{N} \ln\lambda_i = -2\sum_{i=1}^{N} \ln\frac{f_\text{mix}(x_i)}{f(x_i|b)}$$

Under $H_0$ (background only), $t$ follows an asymptotic $\chi^2$ distribution.  We use a Monte Carlo simulation to:
- generate many experiments under $H_0$ and $H_1$,
- build the distributions of $t$, and
- compute the observed $p$-value from the $H_0$ distribution.

This approach is inspired by the reference implementation at:  
https://www.pp.rhul.ac.uk/~cowan/stat/exercises/hypTest/hypTestMC.ipynb

In [ ]:
# -------------------------------------------------------
# PDFs and mixture
# -------------------------------------------------------
def pdf_sig(x):
    """f(x|s) = 3(1-x)^2"""
    return 3.0 * (1.0 - x)**2


def pdf_bkg(x):
    """f(x|b) = 3x^2"""
    return 3.0 * x**2


def pdf_mix(x, p_s, p_b):
    """Mixture PDF under signal+background hypothesis."""
    return p_s * pdf_sig(x) + p_b * pdf_bkg(x)


# -------------------------------------------------------
# Sampling helpers  (inverse CDF method)
# -------------------------------------------------------
def sample_sig(n, rng_):
    """Draw n samples from f(x|s) = 3(1-x)^2 via inverse CDF.
    CDF: F(x) = 1 - (1-x)^3  =>  x = 1 - (1-u)^(1/3)
    """
    u = rng_.uniform(size=n)
    return 1.0 - (1.0 - u) ** (1.0 / 3.0)


def sample_bkg(n, rng_):
    """Draw n samples from f(x|b) = 3x^2 via inverse CDF.
    CDF: F(x) = x^3  =>  x = u^(1/3)
    """
    u = rng_.uniform(size=n)
    return u ** (1.0 / 3.0)


# -------------------------------------------------------
# Test statistic  t = -2 * sum_i ln[f_mix(x_i)/f_bkg(x_i)]
# -------------------------------------------------------
def test_statistic(x_vals, p_s, p_b):
    """Log-likelihood ratio test statistic for an array of x values."""
    log_lr = np.log(pdf_mix(x_vals, p_s, p_b)) - np.log(pdf_bkg(x_vals))
    return -2.0 * np.sum(log_lr)


# -------------------------------------------------------
# Monte Carlo settings
# -------------------------------------------------------
n_s_total = 10    # total expected signal events
n_b_total = 100   # total expected background events
n_events  = n_s_total + n_b_total  # fixed total sample size per experiment
n_exp     = 50_000                 # number of Monte Carlo experiments

p_s = n_s_total / n_events
p_b = n_b_total / n_events

# -------------------------------------------------------
# Generate t under H0 (background only)
# -------------------------------------------------------
t_H0 = np.empty(n_exp)
for i in range(n_exp):
    x_bkg = sample_bkg(n_events, rng)
    t_H0[i] = test_statistic(x_bkg, p_s, p_b)

# -------------------------------------------------------
# Generate t under H1 (signal + background)
# -------------------------------------------------------
t_H1 = np.empty(n_exp)
for i in range(n_exp):
    x_s = sample_sig(n_s_total, rng)
    x_b = sample_bkg(n_b_total, rng)
    x_all = np.concatenate([x_s, x_b])
    t_H1[i] = test_statistic(x_all, p_s, p_b)

print(f"H0: mean(t) = {t_H0.mean():.2f}, std(t) = {t_H0.std():.2f}")
print(f"H1: mean(t) = {t_H1.mean():.2f}, std(t) = {t_H1.std():.2f}")

In [ ]:
# -------------------------------------------------------
# Simulate one "observed" experiment under H1
# and compute its p-value from the H0 distribution
# -------------------------------------------------------
x_obs_s = sample_sig(n_s_total, rng)
x_obs_b = sample_bkg(n_b_total, rng)
x_obs   = np.concatenate([x_obs_s, x_obs_b])
t_obs   = test_statistic(x_obs, p_s, p_b)

# p-value: fraction of H0 experiments with t <= t_obs
# (small t => more signal-like => reject H0)
p_val = np.mean(t_H0 <= t_obs)
Z_obs = norm.isf(p_val) if p_val > 0 else np.inf

print(f"Observed test statistic  t_obs = {t_obs:.3f}")
print(f"p-value (from H0 MC)           = {p_val:.4e}")
print(f"Significance                   = {Z_obs:.2f} sigma")

# Median significance over many H1 experiments
p_H1 = np.array([np.mean(t_H0 <= t) for t in t_H1])
Z_H1 = norm.isf(np.clip(p_H1, 1e-300, 1 - 1e-15))
print(f"\nMedian significance over H1 experiments = {np.median(Z_H1):.2f} sigma")

In [ ]:
# -------------------------------------------------------
# Plot distributions of t under H0 and H1
# -------------------------------------------------------
fig, ax = plt.subplots(figsize=(8, 5))

bins = np.linspace(min(t_H0.min(), t_H1.min()), max(t_H0.max(), t_H1.max()), 80)

ax.hist(t_H0, bins=bins, density=True, histtype='stepfilled',
        alpha=0.5, color='tomato',    label=r'$H_0$: background only')
ax.hist(t_H1, bins=bins, density=True, histtype='stepfilled',
        alpha=0.5, color='royalblue', label=r'$H_1$: signal + background')

ax.axvline(t_obs, color='black', ls='--', lw=1.5,
           label=f'$t_{{\\rm obs}} = {t_obs:.2f}$')

ax.set_xlabel('Test statistic $t$')
ax.set_ylabel('Probability density')
ax.set_title('Distribution of $t$ under $H_0$ and $H_1$')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Statistical inference with `pyhf`

`pyhf` is the standard Python library for **binned statistical models in HEP**,
built on the *HistFactory* formalism.  It encodes signal and background expectations
in a *workspace* and provides asymptotic and toy-based calculations for:
- **Discovery** – $p$-value for the background-only hypothesis (the $q_0$ test),
- **Upper limits** – 95 % $\textrm{CL}_s$ upper limit on the signal strength $\mu$.

We reuse the counting-experiment numbers from sections 3–6 (signal region defined by
$x < x_{\rm cut} = 0.1$, with $s_{\rm tot} = 10$ signal and $b_{\rm tot} = 100$
background events) and add a **10 % Gaussian uncertainty** on the background yield to
illustrate how systematic uncertainties are handled.

### `pyhf` model anatomy

```
pyhf.simplemodels.uncorrelated_background(
    signal           = [n_s],          # expected signal events per bin
    bkg              = [n_b],          # expected background events per bin
    bkg_uncertainty  = [delta_b],      # absolute (1σ) background uncertainty
)
```

This creates a **one-bin HistFactory model** with:
- A *free* signal-strength parameter $\mu$ (POI),
- A background normalisation nuisance parameter $\gamma$ constrained by a
  Gaussian auxiliary measurement.

In [ ]:
# ── Expected yields in the signal region (x < x_cut = 0.1) ─────────────────
# These efficiency formulas follow directly from integrating the PDFs:
#   signal:     ∫₀^x_cut 3(1-x)² dx = 1 - (1 - x_cut)³
#   background: ∫₀^x_cut 3x² dx     = x_cut³
# (Same as in sections 1-3; reproduced here for clarity.)
x_cut_pyhf = 0.1
s_total_pyhf = 10
b_total_pyhf = 100

eps_s_pyhf = 1.0 - (1.0 - x_cut_pyhf)**3     # signal efficiency
eps_b_pyhf = x_cut_pyhf**3                    # background efficiency

n_s_pyhf = s_total_pyhf * eps_s_pyhf          # expected signal in signal region
n_b_pyhf = b_total_pyhf * eps_b_pyhf          # expected background in signal region
delta_b   = 0.10 * n_b_pyhf                   # 10 % background uncertainty

print(f"Expected signal in SR:      n_s = {n_s_pyhf:.4f}")
print(f"Expected background in SR:  n_b = {n_b_pyhf:.4f}")
print(f"Background uncertainty:   delta = {delta_b:.4f}")

# ── Build the pyhf model ─────────────────────────────────────────────────────
model = pyhf.simplemodels.uncorrelated_background(
    signal          = [n_s_pyhf],
    bkg             = [n_b_pyhf],
    bkg_uncertainty = [delta_b],
)

print(f"\nModel parameter names : {model.config.par_names}")
print(f"Parameter of interest : {model.config.poi_name}")
print(f"Suggested init values : {model.config.suggested_init()}")
print(f"Auxiliary data        : {model.config.auxdata}")

### 8a. Discovery: $p$-value for the background-only hypothesis

The **$q_0$ test statistic** (Cowan *et al.* 2011) is used to test $\mu = 0$ (background only):

$$q_0 = \begin{cases}
-2\ln\dfrac{\mathcal{L}(\mu=0,\,\hat{\hat{\boldsymbol{\nu}}})}
           {\mathcal{L}(\hat{\mu},\,\hat{\boldsymbol{\nu}})} & \hat{\mu} > 0 \\
0 & \hat{\mu} \le 0
\end{cases}$$

The asymptotic $p$-value is $p_0 = 1 - \Phi(\sqrt{q_0})$.
We compute it for two scenarios:
- **Background-only** observed data ($n_{\rm obs} = \lfloor n_b \rfloor$) – expect a large $p$-value.
- **Signal + background** observed data ($n_{\rm obs} = \lfloor n_s + n_b \rfloor$) – expect a small $p$-value.


Note that `pyhf` uses the profile-likelihood $q_0$ test (asymptotic), while section 5 uses a pure Poisson tail count test. These are not identical and the results will be, a priori, slightly different.

In [ ]:
import warnings

def make_data(n_obs, model):
    """Combine observed count with the model's auxiliary data."""
    return pyhf.tensorlib.astensor([n_obs] + model.config.auxdata)


# Scenario A: background-only observation
n_obs_b  = int(round(n_b_pyhf))
data_b   = make_data(n_obs_b, model)

# Scenario B: signal + background observation
n_obs_sb = int(round(n_s_pyhf + n_b_pyhf))
data_sb  = make_data(n_obs_sb, model)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    p0_bkg = pyhf.infer.hypotest(0.0, data_b,  model, test_stat='q0')
    p0_sb  = pyhf.infer.hypotest(0.0, data_sb, model, test_stat='q0')

p0_bkg_f = float(p0_bkg)
p0_sb_f  = float(p0_sb)
Z_bkg = float(norm.isf(p0_bkg_f))
Z_sb  = float(norm.isf(p0_sb_f))

print("Discovery test (q0), 10 % background uncertainty")
print(f"  Background-only (n_obs = {n_obs_b}):")
print(f"    p0 = {p0_bkg_f:.4e}   Z = {Z_bkg:.2f} sigma")
print()
print(f"  Signal+background (n_obs = {n_obs_sb}):")
print(f"    p0 = {p0_sb_f:.4e}   Z = {Z_sb:.2f} sigma")
print()

# Compare with manual Poisson p-value from section 5 (no background uncertainty)
p_manual_sb = float(poisson.sf(n_obs_sb - 1, n_b_pyhf))
Z_manual    = float(norm.isf(p_manual_sb))
print("Manual Poisson p-value from section 5 (no bkg uncertainty):")
print(f"    p  = {p_manual_sb:.4e}   Z = {Z_manual:.2f} sigma")

### 8b. Upper limit on signal strength using the $\textrm{CL}_s$ method

We set a **95 % $\textrm{CL}_s$ upper limit** on the
signal strength $\mu$.  The $\textrm{CL}_s$ criterion (Read 2002) avoids setting limits
tighter than the experiment's sensitivity:

$$\text{CL}_s(\mu) = \frac{p_{s+b}(\mu)}{1 - p_b(\mu)} \quad \le 0.05$$

The **expected limit** is the median upper limit when only background is present.
The $\pm 1\sigma$ and $\pm 2\sigma$ bands show the spread over background-only
pseudo-experiments.

In [ ]:
from pyhf.infer.intervals import upper_limits

# Scan mu values for CLs upper limit (background-only observation)
mu_scan = np.linspace(0.01, 10.0, 100)

with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    obs_ul, exp_ul_bands = upper_limits.upper_limit(
        data_b, model,
        scan=mu_scan,
    )

obs_ul       = float(obs_ul)
exp_ul_minus2, exp_ul_minus1, exp_ul_median, exp_ul_plus1, exp_ul_plus2 = [
    float(v) for v in exp_ul_bands
]

print("95 % CLs upper limit on signal strength mu  (bkg-only observation)")
print(f"  Observed UL  :  mu < {obs_ul:.2f}")
print(f"  Expected UL  :  mu < {exp_ul_median:.2f}")
print(f"  +/- 1 sigma  :  [{exp_ul_minus1:.2f}, {exp_ul_plus1:.2f}]")
print(f"  +/- 2 sigma  :  [{exp_ul_minus2:.2f}, {exp_ul_plus2:.2f}]")
print()
print("Interpretation: the data are consistent with background only,")
print(f"so we exclude signal strengths mu > {obs_ul:.2f} at 95% CL_s.")

# Also show the full CLs scan
with warnings.catch_warnings():
    warnings.simplefilter('ignore')
    cls_vals = np.array([
        float(pyhf.infer.hypotest(mu, data_b, model, test_stat='qtilde'))
        for mu in mu_scan
    ])

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mu_scan, cls_vals, 'royalblue', lw=2, label=r'CL$_s(\mu)$ observed')
ax.axhline(0.05, color='tomato', ls='--', lw=1.5, label='95 % threshold (CL$_s$ = 0.05)')
ax.axvline(obs_ul, color='tomato', ls=':', lw=1.5,
           label=f'Obs. UL: $\\mu < {obs_ul:.2f}$')
ax.fill_betweenx([0, 0.05], exp_ul_minus2, exp_ul_plus2,
                  color='yellow', alpha=0.5, label=r'Exp. $\pm2\sigma$')
ax.fill_betweenx([0, 0.05], exp_ul_minus1, exp_ul_plus1,
                  color='limegreen', alpha=0.6, label=r'Exp. $\pm1\sigma$')
ax.axvline(exp_ul_median, color='black', ls='--', lw=1.5,
           label=f'Exp. UL: $\\mu < {exp_ul_median:.2f}$')
ax.set_xlabel(r'Signal strength $\mu$')
ax.set_ylabel(r'CL$_s$')
ax.set_title('95 % CL$_s$ upper limit scan')
ax.set_ylim(0, 1)
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

### Summary

| Question | Result |
|---|---|
| 1. Critical boundary (α = 5 %) | $x_\text{cut} = \alpha^{1/3} \approx 0.368$ |
| 2. Power at $x_\text{cut} = 0.368$ | $1 - \beta = 1 - (1 - x_\text{cut})^3 \approx 0.748$ |
| 3. Expected selected events (x_cut = 0.1) | $n_s \approx 2.71,\quad n_b \approx 0.10$ |
| 4. Signal purity | $\approx 96.4\%$ |
| 5. $p$-value of $s = 0$ | Poisson tail: $P(N \ge n \mid \mu_b = n_b)$ |
| 6. Optimal $x_\text{cut}$ (max $Z_\text{med}$) | Numerically optimised |
| 7. MC test statistic | Full likelihood-ratio test using all $x$ values |
| 8. `pyhf` discovery & upper limit | $q_0$ discovery + 95 % $\textrm{CL}_s$ upper limit |


## Exercises

1. **Optimising the cut with systematics.** In section 6 the expected significance was
   computed assuming the background prediction is exact. Now add a 10% Gaussian uncertainty
   on the background yield. Use the asymptotic formula from Cowan *et al.* (2011, eq. 97):

   $$
   Z \approx \sqrt{2\left[(n_s+n_b)\ln\!\left(\frac{n_s+n_b}{n_b}\cdot
   \frac{n_b^2+\delta_b^2}{n_b^2+n_s\delta_b^2+\delta_b^2}\right)
   - \frac{n_b^2}{\delta_b^2}\ln\!\left(1+\frac{n_s\delta_b^2}{n_b(n_b+\delta_b^2)}\right)\right]}
   $$

   How does the optimal cut change?

2. **$\textrm{CL}_{s}$ vs. $\textrm{CL}_{s+b}$ limits.** Repeat the upper-limit calculation using $\textrm{CL}_{s+b}$ instead of $\textrm{CL}_{s}$. Compare the two limits for a background-only
   observation. Which method gives tighter limits, and why is $\textrm{CL}_{s}$ preferred in HEP?

3. **Two-bin model.** Extend the `pyhf` model to **two bins**: a signal-enriched bin
   ($x < 0.1$) and a control bin ($0.1 \le x < 0.5$) used to constrain the background
   in-situ. Build the model using `pyhf.Model` with a JSON workspace or with
   `pyhf.simplemodels` and verify that adding the control region improves the
   expected discovery significance.

4. **Toys vs. asymptotics.** Use `pyhf.infer.hypotest` with `calctype='toybased'`
   and compare the toy-based $p_0$ with the asymptotic result from section 8a.
   For which regime (small $n_s/n_b$, large $n_s/n_b$) do the two approaches agree?

5. **From $p$-value to exclusion.** Starting from the MC experiment of section 7,
   compute the $\textrm{CL}_{s}$ upper limit on $\mu$ analytically and compare it with the
   `pyhf` result. How does the limit change as you increase the background uncertainty
   from 0% to 50%?